In [ ]:
import os
import numpy as np
from PIL import Image, ImageFilter
import openslide
from trident.Converter import AnyToTiffConverter
import pandas as pd
import torch
import h5py
from pathlib import Path
from trident import OpenSlideWSI
from trident.patch_encoder_models import encoder_factory

# ---------------------------------------------------------------------------- #
# 1) SPLITTING FUNCTION
# ---------------------------------------------------------------------------- #
def split_paths_by_size(paths: list[str], n_jobs: int) -> list[list[str]]:
    """
    Greedy “bin‐packing” split of `paths` into `n_jobs` bins, balancing total file size.
    Returns a list of length n_jobs, each a list of paths.
    """
    # 1) get sizes
    ps = [(p, os.path.getsize(p)) for p in paths]
    # 2) sort descending by size
    ps.sort(key=lambda x: x[1], reverse=True)
    # 3) init bins
    bins = [[] for _ in range(n_jobs)]
    bin_sizes = [0]*n_jobs
    # 4) assign each file to the currently smallest bin
    for p, sz in ps:
        i = min(range(n_jobs), key=lambda j: bin_sizes[j])
        bins[i].append(p)
        bin_sizes[i] += sz
    return bins

In [ ]:
# ---------------------------------------------------------------------------- #
# 3) FEATURE EXTRACTION (uses precomputed list of perturbed WSIs)
# ---------------------------------------------------------------------------- #
def run_perturbed_feature_extraction(
    perturbed_paths: list[str],
    encoder_name: str,
    target_mag: int,
    patch_size: int,
    base_output_dir: str,
    features_root: str,
    mapping_csv: str,
    device: torch.device = None
):
    """
    perturbed_paths : list of perturbed .tif paths
    encoder_name    : key for encoder_factory(...)
    target_mag      : always 20 for these perturbed slides
    patch_size      : patch dimension
    base_output_dir : contains contours_geojson/ and patches/
    features_root   : where to save features
    mapping_csv     : CSV with sample_id,status
    """
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    dev_str = f"{device.type}:{device.index or 0}"

    mapping = pd.read_csv(mapping_csv, dtype={'sample_id':str}).set_index('sample_id')
    encoder = encoder_factory(encoder_name).eval().to(device)

    geojson_dir = Path(base_output_dir)/"contours_geojson"
    coords_dir  = Path(base_output_dir)/"patches"

    for tif_path in perturbed_paths:
        stem = Path(tif_path).stem              # e.g. test_001_brightness_3.333
        parts = stem.split("_",2)
        if len(parts)!=3:
            print("Skipping:", stem); continue
        sample_id, pert, mag_str = parts
        print(f"[feat] {pert}={mag_str} on {sample_id}")

        if sample_id not in mapping.index:
            raise KeyError(sample_id)
        slide_class = int(mapping.at[sample_id,'status'])

        geojson_p = geojson_dir/f"{sample_id}.geojson"
        coords_p  = coords_dir /f"{sample_id}_patches.h5"
        if not geojson_p.exists() or not coords_p.exists():
            raise FileNotFoundError(f"missing base data for {sample_id}")

        slide = OpenSlideWSI(slide_path=str(tif_path), lazy_init=False)
        save_dir = Path(features_root)/encoder_name/pert/mag_str
        save_dir.mkdir(parents=True, exist_ok=True)

        feats_file = slide.extract_patch_features(
            patch_encoder=encoder,
            coords_path=str(coords_p),
            save_features=str(save_dir),
            device=device
        )
        # tag with label
        with h5py.File(feats_file,'a') as h5:
            h5.attrs['label'] = slide_class

        print("  ↳ saved", feats_file)

    print("\n✅ Feature extraction done.")

In [ ]:
# A) Perturbation step in 4 parallel jobs:
PATH = "../../.."
# PATH = "storage"
job_id = 2

wsi_dir  = os.path.join(PATH, "data", "CAMELYON", "test")
all_wsis = sorted(os.path.join(wsi_dir,f) 
                  for f in os.listdir(wsi_dir) if f.lower().endswith(".tif"))
bins     = split_paths_by_size(all_wsis, n_jobs=4)

In [ ]:
# B) Feature extraction step in 4 parallel jobs:
# List all perturbed .tif under perturbed_images
pert_root = os.path.join(wsi_dir,"perturbed_images")
all_pert = sorted(str(p) for p in Path(pert_root).rglob("*.tif"))
bins_feat = split_paths_by_size(all_pert, n_jobs=4)

In [ ]:
# For the same job 2:
run_perturbed_feature_extraction(
    perturbed_paths = bins_feat[job_id],
    encoder_name    = "resnet50",
    target_mag      = 20,
    patch_size      = 512,
    base_output_dir = os.path.join(wsi_dir, "feats"),       # your original coords+geojson
    features_root   = "features",
    mapping_csv     = "mapping.csv",
    device          = torch.device("cuda:0")
)